# DataMate — Kaggle GPU smoke test

Mục tiêu: xác nhận pipeline chạy được với LLM thật trên GPU Kaggle (T4 x2).
Đây **không** phải bài đo 6 giờ; máy chấm là 2 x RTX 5090.

Cài đặt notebook trước khi chạy (Settings bên phải):
- Accelerator: **GPU T4 x2**
- Internet: **On** (chỉ cần cho cell cài đặt / clone; tắt được ở cell suy luận)
- Persistence: Files only
- Input: add dataset chứa trọng số `Qwen2.5-Coder-7B-Instruct-AWQ` (xem cell 2 nếu chưa có)

In [ ]:
# Cell 1 — lấy source code MỚI NHẤT (clone lần đầu; nếu thư mục đã có thì fetch + reset về origin/BRANCH)
import os, subprocess, pathlib
REPO_URL = "https://github.com/HoL0ngg/Zalo-AI-Challenge-2026.git"
BRANCH = "main"
ROOT = pathlib.Path("/kaggle/working/datamate")

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=True).stdout.strip()

if (ROOT / ".git").exists():
    git("remote", "set-url", "origin", REPO_URL, cwd=ROOT)
    git("fetch", "--depth", "1", "origin", BRANCH, cwd=ROOT)
    git("reset", "--hard", "FETCH_HEAD", cwd=ROOT)      # bỏ thay đổi cục bộ, lấy đúng code trên GitHub
    git("clean", "-fd", "-e", "models/", cwd=ROOT)       # giữ symlink trọng số ở models/
else:
    if ROOT.exists():
        import shutil; shutil.rmtree(ROOT)               # thư mục dở dang, không phải repo
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT))
os.chdir(ROOT)
remote = git("ls-remote", REPO_URL, BRANCH).split()[0]   # commit thật sự đang ở đầu nhánh trên GitHub
local = git("rev-parse", "HEAD", cwd=ROOT)
print("local :", git("log", "--oneline", "-1", cwd=ROOT))
print("remote:", remote[:7], "(đầu nhánh", BRANCH, "trên GitHub)")
assert local == remote, "Code chưa mới nhất: kiểm tra đã git push chưa và BRANCH đúng chưa"
# P0 (sql_parser) và data parallel (dataparallel) phải có, nếu không là đang chạy code cũ.
for needed in ("src/datamate/codegen/sql_parser.py", "src/datamate/runtime/dataparallel.py"):
    assert (ROOT / needed).exists(), f"Thiếu {needed}: chưa push code lên GitHub"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 1b — tìm bộ dữ liệu chính thức (train có đáp án, public_test không có đáp án)
# Cần Kaggle Dataset bạn đã upload: mỗi gói là một thư mục tên `train` / `public_test`, chứa questions.jsonl + rulebook.md + calendar.csv + data/.
import pathlib

def find_packages(root="/kaggle/input"):
    found = {}
    for q in pathlib.Path(root).rglob("questions.jsonl"):
        d = q.parent
        if (d / "rulebook.md").exists() and (d / "data").is_dir():
            found[d.name] = d
    return found

pk = find_packages()
# Không có fallback sang dataset/ của repo: repo chỉ chứa dữ liệu tổng hợp cũ, chạy nhầm sẽ cho số vô nghĩa.
assert "train" in pk and "public_test" in pk, (
    "Không thấy Kaggle Dataset chính thức. Add Dataset (chứa thư mục train/ và public_test/) làm Input rồi chạy lại. "
    f"Tìm thấy: {sorted(pk)}")
TRAIN_DIR, TEST_DIR = pk["train"], pk["public_test"]
TRAIN_DIR, TEST_DIR = pathlib.Path(TRAIN_DIR), pathlib.Path(TEST_DIR)
for name, d in (("train", TRAIN_DIR), ("public_test", TEST_DIR)):
    print(name, "->", d, "| questions:", sum(1 for _ in open(d / "questions.jsonl", encoding="utf-8")) if (d / "questions.jsonl").exists() else "MISSING")
assert (TRAIN_DIR / "answers.jsonl").exists(), "train phải có answers.jsonl"
assert not (TEST_DIR / "answers.jsonl").exists(), "public_test không được có đáp án"
FULL_TRAIN = True         # cell 8b chạy đủ 3000 câu train trên 2 GPU (đặt False để chạy mẫu)
SAMPLE_PER_BUSINESS = 8   # train có 3000 câu: mặc định chạy mẫu 8 câu/business (30 business = 240 câu). Đặt None để chạy hết.
SAMPLE_ARG = f"--max-per-business {SAMPLE_PER_BUSINESS}" if SAMPLE_PER_BUSINESS else ""   # cell 5 và 7 (1 GPU, kiểm tra nhanh)
FULL_ARG = "" if FULL_TRAIN else SAMPLE_ARG                                              # cell 8b (2 GPU)

In [ ]:
# Cell 2 — trọng số mô hình.
# Lần đầu: tải từ Hugging Face (Internet On), rồi Save Version -> Output -> "New Dataset"
# để các lần sau add làm Input và chạy offline.
# Tự tìm thư mục chứa config.json + *.safetensors dưới /kaggle/input (Kaggle có thể đặt
# dataset ở /kaggle/input/<slug>/ hoặc /kaggle/input/datasets/<user>/<slug>/, có hoặc không có thư mục con).
import pathlib
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"

def find_weights(root="/kaggle/input"):
    hits = []
    for cfg in pathlib.Path(root).rglob("config.json"):
        folder = cfg.parent
        if any(folder.glob("*.safetensors")) and (folder / "tokenizer.json").exists():
            hits.append(folder)
    return sorted(hits, key=lambda d: len(str(d)))

found = find_weights()
if found:
    weights = found[0]
    print("Dùng trọng số có sẵn:", weights)
    if len(found) > 1:
        print("Các thư mục khác cũng khớp:", found[1:])
else:
    weights = pathlib.Path("/kaggle/working/Qwen2.5-Coder-7B-Instruct-AWQ")
    if not (weights / "config.json").exists():
        from huggingface_hub import snapshot_download
        snapshot_download(MODEL_ID, local_dir=str(weights))
    print("Đã tải trọng số về:", weights, "— hãy Save Version và tạo Dataset từ Output")

link = ROOT / "models" / "qwen-coder-awq"          # phải khớp local_path trong configs/models_kaggle.yaml
link.parent.mkdir(exist_ok=True)
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(weights, target_is_directory=True)
assert (link / "config.json").exists(), f"Symlink trọng số chưa đúng: {link} -> {weights}"
print("models/qwen-coder-awq ->", link.resolve())
print(sorted(f.name for f in weights.iterdir())[:20])

In [ ]:
# Cell 3 — cài phụ thuộc (5–10 phút). vLLM kéo theo bản torch của riêng nó (hiện là cu130).
%pip install -q -e ".[test]"
%pip install -q rapidfuzz Unidecode sqlglot
%pip install -q vllm
# Kaggle cài sẵn torchaudio/torchvision build cho CUDA cũ hơn; lệch CUDA với torch mới sẽ làm
# transformers crash khi import. Dự án không dùng audio nên gỡ torchaudio; torchvision chỉ giữ
# nếu đúng bản CUDA của torch.
%pip uninstall -q -y torchaudio
import sys
import torch
# torchvision phải import được VÀ khớp bản torch: vLLM >= 0.32 import torchvision ở bước warmup kernel
# (thiếu là "No module named 'torchvision'" khi khởi động engine). Không so chuỗi version CUDA:
# torchvision.version.cuda = "13000" còn torch.version.cuda = "13.0" nên luôn lệch.
def torchvision_ok():
    import importlib
    try:
        for name in [m for m in sys.modules if m == "torchvision" or m.startswith("torchvision.")]:
            del sys.modules[name]
        importlib.import_module("torchvision.ops")   # nạp extension C++, lỗi nếu lệch torch/CUDA
        from torchvision.transforms import InterpolationMode  # noqa: F401
        return True
    except Exception as exc:  # noqa: BLE001
        print("torchvision không dùng được:", repr(exc)[:200])
        return False

if not torchvision_ok():
    minor = int(torch.__version__.split(".")[1])          # torch 2.N <-> torchvision 0.(N+15)
    tag = "cu" + torch.version.cuda.replace(".", "")
    !{sys.executable} -m pip uninstall -q -y torchvision
    !{sys.executable} -m pip install -q --no-deps "torchvision==0.{minor + 15}.*" --index-url https://download.pytorch.org/whl/{tag}
assert torchvision_ok(), "torchvision vẫn lỗi: vLLM sẽ không khởi động được engine"
import torchvision
print("torchvision", torchvision.__version__)
import vllm
print("vllm", vllm.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
import transformers.audio_utils  # phải import được mà không lỗi
print("transformers", transformers.__version__, "OK")

In [ ]:
# Cell 4 — kiểm tra CPU/mock trước khi đụng GPU
!python -m pytest -q
!python -m datamate.cli run --data-dir data/samples --result-dir /kaggle/working/result/sample
!python -m datamate.cli eval /kaggle/working/result/sample/submission.jsonl data/samples/answers.jsonl

In [ ]:
# Cell 5 — chạy LLM thật trên tập train (có answers.jsonl để chấm). Từ cell này có thể tắt Internet.
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
import time; _t0 = time.time()
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir {TRAIN_DIR} --result-dir /kaggle/working/result/train {SAMPLE_ARG}
WALL_1GPU = time.time() - _t0
import pathlib
sub = pathlib.Path("/kaggle/working/result/train/submission.jsonl")
assert sub.exists() and sub.stat().st_size > 0, "Chạy LLM lỗi, không có submission: đọc traceback ở trên (thường là vLLM không khởi động)"
open("/kaggle/working/wall_1gpu.txt", "w").write(str(WALL_1GPU))
print(f"1 GPU: {WALL_1GPU:.0f}s")

In [ ]:
# Cell 6 — chấm điểm train + báo cáo theo bước (SỐ BASELINE: gửi file report_train.json)
R = "/kaggle/working/result/train"
!python -m datamate.cli validate {R}/submission.jsonl
!python -m datamate.cli eval {R}/submission.jsonl {TRAIN_DIR}/answers.jsonl --logged-only
!python -m datamate.cli report {R} {TRAIN_DIR}/answers.jsonl > /kaggle/working/report_train.json
!head -c 2500 /kaggle/working/report_train.json

In [ ]:
# Cell 7 (tuỳ chọn) — chạy lần 2 để kiểm tra tính tái lập (yêu cầu lệch <= 0.03)
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir {TRAIN_DIR} --result-dir /kaggle/working/result/train_rerun {SAMPLE_ARG}
!python -m datamate.cli eval /kaggle/working/result/train_rerun/submission.jsonl {TRAIN_DIR}/answers.jsonl --logged-only
!diff <(sort /kaggle/working/result/train/submission.jsonl) <(sort /kaggle/working/result/train_rerun/submission.jsonl) | head -20

## 2 GPU (data parallel)
Mỗi GPU một tiến trình `datamate run` với cùng model (`tensor_parallel_size: 1`), mỗi tiến trình lo một nhóm business rời nhau (`src/datamate/runtime/dataparallel.py`). Vẫn là pipeline tuần tự trong từng tiến trình, nên đây là cách rẻ nhất để dùng cả hai card; kết quả được gộp lại đúng thứ tự `questions.jsonl`. Chạy sau cell 5 để có mốc 1 GPU so sánh thời gian.

In [ ]:
# Cell 8a — kiểm tra có đủ 2 GPU
import torch
n_gpu = torch.cuda.device_count()
print("GPU nhìn thấy:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu >= 2, "Bật Accelerator = GPU T4 x2 trong Settings"

## Nộp thử: public_test (không có đáp án, nên không chấm được ở đây)
Chạy toàn bộ `public_test` trên 2 GPU, kiểm tra định dạng, rồi chép ra `/kaggle/working/submission.jsonl` để tải về và nộp. Đặt **trước** cell chạy đủ train để có file nộp sớm. Ước tính thời gian trên 2×T4: public_test ≈ 40 phút, train 3000 câu ≈ 2 giờ.

In [ ]:
# Cell 8d — public_test (1000 câu, không đáp án), 2 GPU -> /kaggle/working/submission.jsonl  (chạy TRƯỚC train đầy đủ)
import os, time, shutil, json, pathlib
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
RT = "/kaggle/working/result/public_test"
_t0 = time.time()
!python -m datamate.runtime.dataparallel --config configs/kaggle.yaml     --data-dir {TEST_DIR} --result-dir {RT} --gpus 0,1
print(f"public_test: {time.time() - _t0:.0f}s")
_bad = [p.name for p in sorted(pathlib.Path(RT, ".shards").glob("shard_*.log")) if "Traceback" in p.read_text(errors="ignore")]
assert not _bad, f"Shard lỗi: {_bad} — đọc log trong {RT}/.shards/"
!python -m datamate.cli validate {RT}/submission.jsonl
ids = {json.loads(l)["id"] for l in open(f"{TEST_DIR}/questions.jsonl", encoding="utf-8")}
rows = [json.loads(l) for l in open(f"{RT}/submission.jsonl", encoding="utf-8")]
missing = ids - {r["id"] for r in rows}
from collections import Counter
print("câu trả lời:", len(rows), "/", len(ids), "| thiếu (tính abstain):", len(missing), "|", dict(Counter(r["type"] for r in rows)))
shutil.copy(f"{RT}/submission.jsonl", "/kaggle/working/submission.jsonl")

In [ ]:
# Cell 8b — TOÀN BỘ train (3000 câu) trên 2 GPU, chấm theo luật +1/-1 (gửi report_train_2gpu.json)
import os, time
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
R2 = "/kaggle/working/result/train_2gpu"
_t0 = time.time()
!python -m datamate.runtime.dataparallel --config configs/kaggle.yaml     --data-dir {TRAIN_DIR} --result-dir {R2} --gpus 0,1 {FULL_ARG}
WALL_2GPU = time.time() - _t0
import json as _json, pathlib
_sub = pathlib.Path(R2) / "submission.jsonl"
assert _sub.exists() and _sub.stat().st_size > 0, f"Không có kết quả: đọc {R2}/.shards/shard_*.log"
_logs = sorted(pathlib.Path(R2, ".shards").glob("shard_*.log"))
_bad = [p.name for p in _logs if "Traceback" in p.read_text(errors="ignore")]
assert not _bad, f"Shard lỗi: {_bad} — đọc log trong {R2}/.shards/"
try:
    w1 = float(open("/kaggle/working/wall_1gpu.txt").read())
    print(f"1 GPU {w1:.0f}s | 2 GPU {WALL_2GPU:.0f}s | tăng tốc x{w1 / WALL_2GPU:.2f} (gồm cả thời gian nạp model)")
except FileNotFoundError:
    print(f"2 GPU {WALL_2GPU:.0f}s (chưa có mốc 1 GPU: chạy cell 5 trước)")
!python -m datamate.cli validate {R2}/submission.jsonl
!python -m datamate.cli eval {R2}/submission.jsonl {TRAIN_DIR}/answers.jsonl --logged-only
!python -m datamate.cli report {R2} {TRAIN_DIR}/answers.jsonl > /kaggle/working/report_train_2gpu.json
!head -c 1500 /kaggle/working/report_train_2gpu.json
# Nếu một shard lỗi: đọc {R2}/.shards/shard_*.log

In [ ]:
# Cell 8c (tuỳ chọn) — 1 GPU (mẫu) và 2 GPU (đủ train): các câu chung phải cho cùng đáp án
import json
def load(p): return {r["id"]: r for r in map(json.loads, open(p, encoding="utf-8"))}
one, two = load("/kaggle/working/result/train/submission.jsonl"), load(R2 + "/submission.jsonl")
common = [i for i in one if i in two]
diff = [i for i in common if one[i] != two[i]]
print(f"khác nhau: {len(diff)}/{len(common)} câu chung", diff[:10])

## Đo thêm
- **Số token thật của prompt** (cần tokenizer của model, chạy sau cell 2):
  `!python scripts/measure_prompts.py --data-dir dataset/train --tokenizer models/qwen-coder-awq`
- **So với trước P0** (tuỳ chọn): chạy 2 cell tiếp theo.

In [ ]:
# Cell 11 — đóng gói kết quả để tải về (Output -> datamate_results.zip) và gửi cho mình
!cd /kaggle/working && zip -rq datamate_results.zip report_*.json submission.jsonl result -x "result/sample/*" "result/*/.shards/in_*"
!ls -la /kaggle/working/datamate_results.zip